# 🧠 Notebook 3 — LSTM Forecasting

LSTM (Long Short-Term Memory) is a recurrent neural network capable of learning complex temporal dependencies.
We treat sales forecasting as a **supervised learning** problem using a sliding window approach.

## 3.1 Setup

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.join('..', 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_preprocessing import (load_and_clean, aggregate_monthly,
                                 train_test_split_ts, create_lstm_sequences, scale_series)
from lstm_model import (build_lstm_model, train_lstm, predict_lstm,
                        plot_lstm_training, plot_lstm_forecast)
from evaluate import evaluate_model

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

os.makedirs('../outputs', exist_ok=True)
LOOKBACK = 12   # Use last 12 months to predict next month


## 3.2 Load & Split

In [ ]:
df = load_and_clean('../data/train.csv')
monthly = aggregate_monthly(df)

TEST_MONTHS = 6
train_df, test_df = train_test_split_ts(monthly, test_months=TEST_MONTHS)

train_vals = train_df['y'].values
test_vals  = test_df['y'].values
print(f"Train points: {len(train_vals)} | Test points: {len(test_vals)}")


## 3.3 Normalize Data

LSTMs are sensitive to input scale. We use MinMaxScaler fitted **only on train** to avoid data leakage.

In [ ]:
train_scaled, test_scaled, scaler = scale_series(train_vals, test_vals)
print(f"Train range after scaling: [{train_scaled.min():.3f}, {train_scaled.max():.3f}]")
print(f"Test range after scaling:  [{test_scaled.min():.3f}, {test_scaled.max():.3f}]")


## 3.4 Create Sequences

We use a **sliding window** of `LOOKBACK=12` months to create (X, y) pairs.  
Each X is 12 consecutive months; y is the next month's sales.

In [ ]:
full_scaled = np.concatenate([train_scaled, test_scaled])
split_idx   = len(train_scaled)

X_all, y_all = create_lstm_sequences(full_scaled, lookback=LOOKBACK)

X_train = X_all[:split_idx - LOOKBACK]
y_train = y_all[:split_idx - LOOKBACK]
X_test  = X_all[split_idx - LOOKBACK:]
y_test  = y_all[split_idx - LOOKBACK:]

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   |  y_test:  {y_test.shape}")


## 3.5 Build LSTM Architecture

In [ ]:
model = build_lstm_model(
    lookback=LOOKBACK,
    lstm_units_1=64,
    lstm_units_2=32,
    dropout_rate=0.2,
    learning_rate=0.001
)


## 3.6 Train LSTM

In [ ]:
history = train_lstm(model, X_train, y_train, epochs=150, batch_size=4)


## 3.7 Training Loss Curve

In [ ]:
plot_lstm_training(history, save_path='../outputs/lstm_training_history.png')


## 3.8 Predictions on Test Set

In [ ]:
lstm_preds = predict_lstm(model, X_test, scaler)
test_actual = scaler.inverse_transform(y_test.reshape(-1,1)).flatten()

result = evaluate_model('LSTM', test_actual, lstm_preds)

comparison = test_df[['ds']].iloc[-len(lstm_preds):].copy()
comparison['Actual'] = test_actual
comparison['LSTM_Predicted'] = lstm_preds.round(2)
comparison['Error'] = (comparison['Actual'] - comparison['LSTM_Predicted']).round(2)
comparison['Error_%'] = (comparison['Error'] / comparison['Actual'] * 100).round(2)
comparison


## 3.9 Visualize Predictions

In [ ]:
plot_lstm_forecast(
    train_df['ds'], train_vals,
    test_df['ds'].iloc[-len(lstm_preds):], test_actual,
    lstm_preds,
    save_path='../outputs/lstm_forecast.png'
)


## 3.10 Future Recursive Forecast (Beyond Test Set)

In [ ]:
from lstm_model import recursive_forecast

last_seq = train_scaled[-LOOKBACK:]
future_preds = recursive_forecast(model, last_seq, n_steps=6, scaler=scaler)

import pandas as pd
future_dates = pd.date_range(start=monthly['ds'].max(), periods=7, freq='MS')[1:]
future_df = pd.DataFrame({'Date': future_dates, 'Forecasted_Sales': future_preds.round(2)})
print("\n6-Month Future Forecast:")
print(future_df.to_string(index=False))


## 3.11 LSTM — Pros & Cons

| ✅ Pros | ❌ Cons |
|---|---|
| Captures non-linear temporal patterns | Black box — hard to explain predictions |
| Can model long-range dependencies | Needs more data for reliable training |
| Flexible architecture | Slow to train, requires hyperparameter tuning |
| Can be extended with multivariate inputs | Sensitive to scaling and initialization |